# LeetCode #1115: Print FooBar Alternately

https://leetcode.com/problems/print-foobar-alternately/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin loop on shared flag | Wastes CPU |
| **Optimal: Two Semaphores Alternating ★** | `fooSem(1)` + `barSem(0)` | Each thread releases the other's gate after printing |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Two Semaphores Alternating ★
Initialize `fooSem` to 1 (foo goes first) and `barSem` to 0. The foo thread acquires `fooSem`, prints, releases `barSem`. The bar thread acquires `barSem`, prints, releases `fooSem`. They alternate perfectly for `n` iterations.

**Constraints:**
* Two threads call `foo()` and `bar()` concurrently
* Output must always be `foobarfoobar...` exactly `n` times
* `1 <= n <= 1000`


## Solutions
### C#

In [ ]:
using System.Threading;

public class FooBar
{
    private readonly int _n;
    private readonly SemaphoreSlim _fooSem = new SemaphoreSlim(1, 1);
    private readonly SemaphoreSlim _barSem = new SemaphoreSlim(0, 1);

    public FooBar(int n) => _n = n;

    public void Foo(Action printFoo)
    {
        for (int i = 0; i < _n; i++)
        {
            _fooSem.Wait();
            printFoo();
            _barSem.Release();
        }
    }

    public void Bar(Action printBar)
    {
        for (int i = 0; i < _n; i++)
        {
            _barSem.Wait();
            printBar();
            _fooSem.Release();
        }
    }
}

### Python

In [ ]:
import threading

class FooBar:
    def __init__(self, n: int):
        self.n = n
        self.foo_sem = threading.Semaphore(1)  # foo goes first
        self.bar_sem = threading.Semaphore(0)

    def foo(self, printFoo: 'Callable[[], None]') -> None:
        for _ in range(self.n):
            self.foo_sem.acquire()
            printFoo()
            self.bar_sem.release()

    def bar(self, printBar: 'Callable[[], None]') -> None:
        for _ in range(self.n):
            self.bar_sem.acquire()
            printBar()
            self.foo_sem.release()

### Go

In [ ]:
package main

type FooBar struct {
	n      int
	fooGo  chan struct{}
	barGo  chan struct{}
}

func NewFooBar(n int) *FooBar {
	fb := &FooBar{
		n:     n,
		fooGo: make(chan struct{}, 1),
		barGo: make(chan struct{}, 1),
	}
	fb.fooGo <- struct{}{} // foo goes first
	return fb
}

func (fb *FooBar) Foo(printFoo func()) {
	for i := 0; i < fb.n; i++ {
		<-fb.fooGo
		printFoo()
		fb.barGo <- struct{}{}
	}
}

func (fb *FooBar) Bar(printBar func()) {
	for i := 0; i < fb.n; i++ {
		<-fb.barGo
		printBar()
		fb.fooGo <- struct{}{}
	}
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex, Condvar};

struct FooBar {
    n: usize,
    state: Mutex<bool>, // true = foo's turn
    cv: Condvar,
}

impl FooBar {
    fn new(n: usize) -> Arc<Self> {
        Arc::new(FooBar { n, state: Mutex::new(true), cv: Condvar::new() })
    }

    fn foo(&self, print_foo: impl Fn()) {
        for _ in 0..self.n {
            let mut turn = self.state.lock().unwrap();
            while !*turn { turn = self.cv.wait(turn).unwrap(); }
            print_foo();
            *turn = false;
            self.cv.notify_all();
        }
    }

    fn bar(&self, print_bar: impl Fn()) {
        for _ in 0..self.n {
            let mut turn = self.state.lock().unwrap();
            while *turn { turn = self.cv.wait(turn).unwrap(); }
            print_bar();
            *turn = true;
            self.cv.notify_all();
        }
    }
}

## Concurrency Scenarios

1. **Both threads start simultaneously**: `barSem` is 0 so bar blocks immediately; foo runs first, releases `barSem`, and they alternate correctly from there.
2. **Foo thread is slow**: Bar blocks on `barSem` waiting for foo to finish each iteration — the semaphore count never exceeds 1, so bar cannot run ahead.
3. **n = 1**: Foo prints once, bar prints once, both exit after one iteration — no deadlock or over-release.
4. **Bar thread scheduled before Foo is ready**: Bar acquires `barSem` (which is 0) and parks; OS reschedules foo, which prints and unparks bar.
5. **n = 1000 high-throughput**: With buffered semaphores of size 1, the producer–consumer handshake ensures strict ordering across all 1000 pairs without busy waiting.
